# 🔍 Multi-Business Lead Finder — L&D Designs
Searches **all business types at once** — barbers, plumbers, electricians, restaurants and more.
Finds every business in a 50-mile radius of Wigan with **no website or outdated website**.

1. Run Cell 1 — installs everything and runs the full search (takes 10–20 mins)
2. Run Cell 2 — downloads your spreadsheet

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 1 — Runs ALL business types. Just hit play and wait.
# ══════════════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'requests', 'beautifulsoup4', 'openpyxl', 'lxml'])

import re, math, time, unicodedata
from datetime import datetime
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Config ─────────────────────────────────────────────────────────────
WIGAN_LAT = 53.5450
WIGAN_LNG = -2.6325
RADIUS_MI = 50
RADIUS_M  = int(RADIUS_MI * 1609.344)
OUTDATED_YEARS = 3
SOCIAL_DOMAINS = ('facebook.com','instagram.com','twitter.com','tiktok.com','linkedin.com','linktree.com')
BROWSER_HDR = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'}
YELL_HDR = {**BROWSER_HDR,'Accept':'text/html,application/xhtml+xml;q=0.9,*/*;q=0.8','Accept-Language':'en-GB,en;q=0.9','Referer':'https://www.yell.com/'}

# ── ALL Business types to search ───────────────────────────────────────
PRESETS = {
    'Barbers & Hair Salons': {
        'yell': ['barbers', 'hairdressers', 'hair salon', 'barbershop'],
        'osm':  [('shop','hairdresser'),('shop','barber'),('amenity','hairdresser')],
    },
    'Plumbers': {
        'yell': ['plumbers', 'plumbing services', 'emergency plumber'],
        'osm':  [('craft','plumber')],
    },
    'Electricians': {
        'yell': ['electricians', 'electrical contractor', 'emergency electrician'],
        'osm':  [('craft','electrician')],
    },
    'Restaurants & Takeaways': {
        'yell': ['restaurant', 'takeaway', 'cafe', 'fish and chips', 'pizza'],
        'osm':  [('amenity','restaurant'),('amenity','fast_food'),('amenity','cafe')],
    },
    'Beauty Salons': {
        'yell': ['beauty salon', 'nail salon', 'nail bar', 'lashes', 'aesthetics'],
        'osm':  [('shop','beauty'),('shop','cosmetics')],
    },
    'Car Garages & MOT': {
        'yell': ['car garage', 'MOT centre', 'car repair', 'mechanic', 'tyre fitting'],
        'osm':  [('shop','car_repair'),('shop','tyres')],
    },
    'Cleaning Companies': {
        'yell': ['cleaning company', 'domestic cleaner', 'office cleaning', 'carpet cleaning'],
        'osm':  [],
    },
    'Landscapers & Gardeners': {
        'yell': ['landscaper', 'gardener', 'garden maintenance', 'tree surgeon'],
        'osm':  [('craft','gardener'),('craft','landscaper')],
    },
    'Tattoo Studios': {
        'yell': ['tattoo studio', 'tattoo parlour', 'piercing studio'],
        'osm':  [('shop','tattoo')],
    },
    'Gyms & Personal Trainers': {
        'yell': ['gym', 'personal trainer', 'fitness studio', 'boxing gym'],
        'osm':  [('leisure','fitness_centre'),('leisure','sports_centre')],
    },
    'Dog Groomers': {
        'yell': ['dog groomer', 'dog grooming', 'pet grooming'],
        'osm':  [('shop','pet_grooming')],
    },
    'Accountants': {
        'yell': ['accountant', 'bookkeeper', 'tax advisor'],
        'osm':  [('office','accountant')],
    },
}

# ── Helper functions ────────────────────────────────────────────────────
def haversine_mi(lat1,lng1,lat2,lng2):
    R=3958.8; p1,p2=math.radians(lat1),math.radians(lat2)
    dp=math.radians(lat2-lat1); dl=math.radians(lng2-lng1)
    a=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def _osm_address(tags):
    parts=[tags.get('addr:housenumber',''),tags.get('addr:street',''),
           tags.get('addr:city','') or tags.get('addr:town',''),tags.get('addr:postcode','')]
    return ', '.join(p for p in parts if p)

def fetch_from_overpass(osm_tags):
    if not osm_tags: return []
    tag_lines=''
    for k,v in osm_tags:
        tag_lines+='node["'+k+'"="'+v+'"](around:'+str(RADIUS_M)+','+str(WIGAN_LAT)+','+str(WIGAN_LNG)+');\n'
        tag_lines+='way["'+k+'"="'+v+'"](around:'+str(RADIUS_M)+','+str(WIGAN_LAT)+','+str(WIGAN_LNG)+');\n'
    query='[out:json][timeout:90];\n(\n'+tag_lines+');\nout center tags;'
    try:
        resp=requests.post('https://overpass-api.de/api/interpreter',data={'data':query},
                           headers={'User-Agent':'WiganLeadFinder/1.0'},timeout=120)
        resp.raise_for_status()
    except Exception as e: print('OSM error:',e); return []
    elements=resp.json().get('elements',[])
    results=[]
    for el in elements:
        tags=el.get('tags',{}); name=tags.get('name','').strip()
        if not name: continue
        if el['type']=='node': blat,blng=el.get('lat'),el.get('lon')
        else: c=el.get('center',{}); blat,blng=c.get('lat'),c.get('lon')
        if blat is None: continue
        dist=haversine_mi(WIGAN_LAT,WIGAN_LNG,blat,blng)
        if dist>RADIUS_MI: continue
        results.append({'name':name,'phone':tags.get('phone') or tags.get('contact:phone',''),
                        'website':tags.get('website') or tags.get('contact:website',''),
                        'address':_osm_address(tags),'rating':'','reviews':0,
                        'distance_mi':round(dist,1),'source':'osm'})
    return results

def fetch_from_yell(yell_keywords):
    all_results=[]
    for keyword in yell_keywords:
        print('  Yell: "'+keyword+'" ',end='',flush=True)
        for page_num in range(1,16):
            try:
                resp=requests.get('https://www.yell.com/ucs/UcsSearchAction.do',
                    params={'keywords':keyword,'location':'Wigan, Greater Manchester','radius':'50','pageNum':str(page_num)},
                    headers=YELL_HDR,timeout=20)
                if resp.status_code!=200: break
                html=resp.text
            except: break
            soup=BeautifulSoup(html,'lxml')
            arts=soup.find_all('article',class_=lambda c:c and 'businessCapsule--listing' in c)
            if not arts: break
            for art in arts:
                def _t(cls): tag=art.find(class_=lambda c:c and cls in c); return tag.get_text(strip=True) if tag else ''
                name=_t('businessCapsule--name')
                if not name: continue
                site_tag=art.find('a',class_=lambda c:c and 'businessCapsule--website' in c)
                website=(site_tag.get('data-url') or site_tag.get('href','')) if site_tag else ''
                dist_txt=_t('businessCapsule--distance')
                dist_mi=0.0
                m=re.search(r'([\d.]+)\s*miles?',dist_txt,re.I)
                if m: dist_mi=float(m.group(1))
                if dist_mi<=RADIUS_MI:
                    all_results.append({'name':name,'phone':_t('businessCapsule--telephone'),
                                        'website':website,'address':_t('businessCapsule--address'),
                                        'rating':'','reviews':0,'distance_mi':dist_mi,'source':'yell'})
            print('.',end='',flush=True)
            if len(arts)<10: break
            time.sleep(1.5)
        print()
    return all_results

_PC_RE=re.compile(r'\b([A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2})\b',re.I)
def _norm(s): s=unicodedata.normalize('NFKD',s).encode('ascii','ignore').decode(); return re.sub(r'[^a-z0-9]','',s.lower())
def _postcode(addr): m=_PC_RE.search(addr); return _norm(m.group(1)) if m else ''

def deduplicate_list(items):
    seen={}; merged=[]
    for biz in items:
        words=biz['name'].split(); first=_norm(words[0]) if words else ''
        pc=_postcode(biz['address']); key=(first,pc) if pc else (_norm(biz['name']),)
        if key not in seen: seen[key]=True; merged.append(biz)
    return merged

def check_website(url):
    if not url: return 'none','No website listed'
    if any(d in url.lower() for d in SOCIAL_DOMAINS):
        d=next(x for x in SOCIAL_DOMAINS if x in url.lower()); return 'social_only','Only a '+d+' page'
    try:
        resp=requests.get(url,headers=BROWSER_HDR,timeout=12,allow_redirects=True)
    except requests.exceptions.SSLError: return 'outdated','Broken SSL'
    except: return 'error','Cannot connect'
    if resp.status_code>=400: return 'error','HTTP '+str(resp.status_code)
    cur=datetime.now().year; text=resp.text
    yrs=re.findall(r'copyright[^\d]{0,10}(\d{4})',text,re.I)+re.findall(r'[©]\s*(\d{4})',text)
    valid=[int(y) for y in yrs if 2000<=int(y)<=cur+1]
    if valid and cur-max(valid)>=OUTDATED_YEARS: return 'outdated','Copyright: '+str(max(valid))
    return 'active','Website looks current'

EMAIL_RE=re.compile(r'\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b')
SKIP_EM={'noreply','no-reply','example','test','wordpress','sentry','privacy','abuse'}

def scrape_email(url):
    if not url or any(d in url.lower() for d in SOCIAL_DOMAINS): return ''
    try:
        r=requests.get(url,headers=BROWSER_HDR,timeout=10,allow_redirects=True)
        if r.status_code!=200: return ''
        for e in EMAIL_RE.findall(r.text):
            if not any(s in e.lower() for s in SKIP_EM) and '.' in e.split('@')[-1]:
                return e.lower()
    except: pass
    return ''

# ── MAIN: Loop through ALL business types ──────────────────────────────
print('Starting multi-type search for all business types...')
print('This will take 10-20 minutes. Grab a coffee!\n')

global_seen = set()
all_raw = []

for type_label, cfg in PRESETS.items():
    print('\n' + '='*55)
    print('Searching:', type_label)
    print('='*55)

    osm = fetch_from_overpass(cfg['osm'])
    yell = fetch_from_yell(cfg['yell'])
    combined = deduplicate_list(osm + yell)

    added = 0
    for biz in combined:
        words = biz['name'].split()
        first = _norm(words[0]) if words else ''
        pc = _postcode(biz['address'])
        key = (first, pc) if pc else (_norm(biz['name']),)
        if key not in global_seen:
            global_seen.add(key)
            biz['business_type'] = type_label
            all_raw.append(biz)
            added += 1

    print('  +'+str(added)+' new unique businesses  |  Total so far: '+str(len(all_raw)))

print('\n\n' + '='*55)
print('Total unique businesses found:', len(all_raw))
print('Now checking websites — this takes a while...')
print('='*55)

leads = []
for idx, biz in enumerate(all_raw, 1):
    status, notes = check_website(biz.get('website',''))
    if idx % 50 == 0:
        print('  Checked '+str(idx)+'/'+str(len(all_raw))+'...')
    if status == 'active':
        continue
    email = ''
    if biz.get('website') and status not in ('none','error'):
        email = scrape_email(biz['website'])
    leads.append({**biz, 'website_status': status, 'status_notes': notes, 'email': email})

print('\nDone! ' + str(len(leads)) + ' leads with no/outdated website')

# ── Save spreadsheet ───────────────────────────────────────────────────
wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Leads'

COLS = [
    ('Business Name', 30), ('Business Type', 24), ('Phone', 18),
    ('Email', 32), ('Website Status', 16), ('Address', 42),
    ('Status Notes', 28), ('Distance (mi)', 14), ('Original Website', 38),
]

FILL_H = PatternFill('solid', fgColor='1A2035')
FILLS = {
    'none':        PatternFill('solid', fgColor='FFD6D6'),
    'social_only': PatternFill('solid', fgColor='D6EAFF'),
    'outdated':    PatternFill('solid', fgColor='FFF2CC'),
    'error':       PatternFill('solid', fgColor='E8E8E8'),
}
THIN = Border(**{s: Side(style='thin', color='CCCCCC') for s in ('left','right','top','bottom')})

for col, (hdr, w) in enumerate(COLS, 1):
    c = ws.cell(1, col, hdr)
    c.fill = FILL_H
    c.font = Font(color='FFFFFF', bold=True, size=10)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = THIN
    ws.column_dimensions[get_column_letter(col)].width = w
ws.row_dimensions[1].height = 28

ORDER = {'none': 0, 'social_only': 1, 'outdated': 2, 'error': 3}
sorted_leads = sorted(leads, key=lambda b: (ORDER.get(b['website_status'], 9), b.get('distance_mi', 99)))

for ri, biz in enumerate(sorted_leads, 2):
    fill = FILLS.get(biz['website_status'], PatternFill(fill_type=None))
    row_vals = [
        biz.get('name',''),
        biz.get('business_type',''),
        biz.get('phone',''),
        biz.get('email',''),
        biz.get('website_status','').upper().replace('_',' '),
        biz.get('address',''),
        biz.get('status_notes',''),
        round(float(biz.get('distance_mi',0)), 1),
        biz.get('website',''),
    ]
    for col, val in enumerate(row_vals, 1):
        c = ws.cell(ri, col, val)
        c.fill = fill
        c.alignment = Alignment(vertical='center')
        c.border = THIN
    ws.row_dimensions[ri].height = 17

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
fname = 'multi_leads_' + ts + '.xlsx'
wb.save(fname)

print('\n✅ Saved:', fname)
print('Total leads:', len(sorted_leads))
print()
by_type = {}
for b in sorted_leads:
    t = b.get('business_type','Unknown')
    by_type[t] = by_type.get(t, 0) + 1
print('Breakdown by type:')
for t, n in sorted(by_type.items(), key=lambda x: -x[1]):
    print('  ', t, ':', n)
print('\nWith phone:', sum(1 for b in sorted_leads if b.get('phone')))
print('With email:', sum(1 for b in sorted_leads if b.get('email')))

In [ ]:
# ── Cell 2: Download ──────────────────────────────────────────────────
from google.colab import files
files.download(fname)
print('Check your Downloads folder for', fname)